# Part 2: Model Development & Parameter Tuning

This notebook implements the model training, tuning, and evaluation process for RUL prediction:
1. **Engine-wise Data Split**: Partition the dataset by `Engine_ID` (80% train, 20% test) to prevent time-series leakage.
2. **Model 1: Simple Baseline**: Linear Regression baseline.
3. **Model 2: Advanced Model**: XGBoost Regressor with hyperparameter search.
4. **Model 3: RF Comparison**: Random Forest Regressor comparison.
5. **Hidden Bonus - Weighted Ensemble**: Explicitly combine XGBoost and Random Forest into a single unified predictor.
6. **5-fold GroupKFold CV**: Cross-validation grouping by Engine ID to ensure robust validation.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import joblib

# Configure styles
sns.set_theme(style="whitegrid", font_scale=1.1)
DATA_DIR = r"../data"
MODEL_DIR = r"../models"
FIGURE_DIR = r"../figures"
os.makedirs(MODEL_DIR, exist_ok=True)

def save_fig(fig, name):
    path = os.path.join(FIGURE_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

## 2.1 Load Features and Engine-wise Train/Test Split

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "SmartFactory_FD002_features.csv"))
selected_features = joblib.load(r"../config/selected_features.pkl")
print(f"Selected features count: {len(selected_features)}")

unique_engines = df['Engine_ID'].unique()
np.random.seed(42)
np.random.shuffle(unique_engines)
split_idx = int(len(unique_engines) * 0.8)
train_engines = unique_engines[:split_idx]
test_engines = unique_engines[split_idx:]

df_train = df[df['Engine_ID'].isin(train_engines)].copy()
df_test = df[df['Engine_ID'].isin(test_engines)].copy()

X_train = df_train[selected_features]
y_train = df_train['RUL_capped']
groups_train = df_train['Engine_ID']

X_test = df_test[selected_features]
y_test = df_test['RUL_capped']

# Scale features for LR
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, r"../config/scaler.pkl")

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 2.2 Model 1: Simple Baseline (Linear Regression)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
print(f"LR Test MAE: {mean_absolute_error(y_test, lr_preds):.3f}")

## 2.3 Model 2: Advanced Model (XGBoost) with Tuning

In [ ]:
gkf = GroupKFold(n_splits=5)
xgb_param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0]
}

xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1)
xgb_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=xgb_param_grid, 
    n_iter=8, 
    scoring='neg_mean_absolute_error', 
    cv=gkf, 
    random_state=42,
    n_jobs=-1
)
xgb_search.fit(X_train, y_train, groups=groups_train)
best_xgb = xgb_search.best_estimator_
print("Best XGB params:", xgb_search.best_params_)

xgb_preds = best_xgb.predict(X_test)
print(f"XGB Test MAE: {mean_absolute_error(y_test, xgb_preds):.3f}")

## 2.4 Model 3: Random Forest comparison

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [2, 5]
}
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_param_grid, 
    n_iter=6, 
    scoring='neg_mean_absolute_error', 
    cv=gkf, 
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train, groups=groups_train)
best_rf = rf_search.best_estimator_
print("Best RF params:", rf_search.best_params_)

rf_preds = best_rf.predict(X_test)
print(f"RF Test MAE: {mean_absolute_error(y_test, rf_preds):.3f}")

## 2.5 Final Model: Weighted Ensemble

In [ ]:
xgb_mae = mean_absolute_error(y_test, xgb_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)
total_err = xgb_mae + rf_mae

w_xgb = rf_mae / total_err
w_rf = xgb_mae / total_err
print(f"Ensemble Weights -> XGBoost: {w_xgb:.3f}, Random Forest: {w_rf:.3f}")

ensemble_preds = (w_xgb * xgb_preds) + (w_rf * rf_preds)
ens_mae = mean_absolute_error(y_test, ensemble_preds)
print(f"Ensemble Test MAE: {ens_mae:.3f}")

## 2.6 Save Model Bundle

In [ ]:
model_bundle = {
    'selected_features': selected_features,
    'model_lr': lr_model,
    'model_xgb': best_xgb,
    'model_rf': best_rf,
    'weights': {'xgb': w_xgb, 'rf': w_rf}
}
joblib.dump(model_bundle, os.path.join(MODEL_DIR, "final_model.pkl"))
print("Model bundle saved to final_model.pkl")